# Notebook 10: Comprehensive Analysis & Diagnostics

**Capstone: Bringing Everything Together**

## Overview

This notebook synthesizes results from all previous analyses:
- Traditional hypothesis tests (confidence intervals, t-tests)
- CUPED variance reduction
- Uplift modeling & heterogeneous treatment effects  
- Multi-armed bandits
- Power analysis & diagnostics

We'll create a master comparison, assess agreement across methods, evaluate assumptions, and develop a framework for recommending when to use each approach.

In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Create output directory
os.makedirs('../data/outputs/nb10', exist_ok=True)


In [2]:
# Load original data
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

# Load results from previous notebooks
print("Loading results from all analyses...\n")

# CUPED results (from notebook 7)
try:
    cuped_results = pd.read_csv('../data/outputs/nb07/nb07_cuped_results.csv')
    print("CUPED results loaded")
except:
    print("CUPED results not found - will be generated")
    cuped_results = None

# Uplift results (from notebook 8)
try:
    uplift_results = pd.read_csv('../data/outputs/nb08/nb08_uplift_results.csv')
    print("Uplift results loaded")
except:
    print("Uplift results not found")
    uplift_results = None

# Bandit results (from notebook 9)
try:
    bandit_results = pd.read_csv('../data/outputs/nb09/nb09_bandit_results.csv')
    print("Bandit results loaded")
except:
    print("Bandit results not found")
    bandit_results = None

print(f"\nDataset: {len(df)} customers across 3 segments")
print(f"\nSegment breakdown:")
print(df['segment'].value_counts())

Loading results from all analyses...

CUPED results loaded
Uplift results loaded
Bandit results loaded

Dataset: 64000 customers across 3 segments

Segment breakdown:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


## Method Comparison Dashboard

We'll create a master table comparing conclusions across all methods for the main outcome (conversion).

Each method provides:
- Test statistic (t-stat, p-value, Qini coefficient, etc.)
- P-value or posterior probability
- Effect size
- Confidence interval or credible interval
- Conclusion (Significant? Yes/No)

In [3]:
# Create master comparison table for conversion outcome
# Focus on Mens Email vs No Email treatment

# Prepare data
men_email = df[df['segment'] == 'Mens E-Mail']
control = df[df['segment'] == 'No E-Mail']

y_treat = men_email['conversion'].astype(float)
y_control = control['conversion'].astype(float)

# 1. Traditional t-test (two-tailed)
t_stat, p_value_t = stats.ttest_ind(y_treat, y_control)
effect_size_t = (y_treat.mean() - y_control.mean())
se_t = np.sqrt(y_treat.var()/len(y_treat) + y_control.var()/len(y_control))
ci_t = (effect_size_t - 1.96*se_t, effect_size_t + 1.96*se_t)

# 2. Welch's t-test (doesn't assume equal variances)
t_stat_w, p_value_w = stats.ttest_ind(y_treat, y_control, equal_var=False)

# 3. Mann-Whitney U (non-parametric alternative)
u_stat, p_value_u = stats.mannwhitneyu(y_treat, y_control, alternative='two-sided')

# 4. Chi-square for proportions (binary outcome)
table = np.array([
    [y_treat.sum(), (1-y_treat).sum()],
    [y_control.sum(), (1-y_control).sum()]
])
chi2, p_value_chi2, dof, expected = stats.chi2_contingency(table)

# 5. Fisher's exact test (more conservative for small counts)
from scipy.stats import fisher_exact
odds_ratio, p_value_fisher = fisher_exact(table)

# 6. Bayesian analysis with conjugate Beta-Binomial
alpha_prior = 1
beta_prior = 1

# Posterior for treatment
alpha_post_t = alpha_prior + y_treat.sum()
beta_post_t = beta_prior + (1 - y_treat).sum()

# Posterior for control  
alpha_post_c = alpha_prior + y_control.sum()
beta_post_c = beta_prior + (1 - y_control).sum()

# Bayesian credible interval
from scipy.stats import beta as beta_dist
ci_bayes_lower_t = beta_dist.ppf(0.025, alpha_post_t, beta_post_t)
ci_bayes_upper_t = beta_dist.ppf(0.975, alpha_post_t, beta_post_t)

# Probability that treatment > control
samples_t = np.random.beta(alpha_post_t, beta_post_t, 10000)
samples_c = np.random.beta(alpha_post_c, beta_post_c, 10000)
prob_treat_better = (samples_t > samples_c).mean()

# Create comparison dataframe
comparison_methods = pd.DataFrame({
    'Method': [
        'Traditional t-test',
        "Welch's t-test", 
        'Mann-Whitney U',
        'Chi-square',
        "Fisher's Exact",
        'Bayesian Beta-Binomial'
    ],
    'Test Stat': [
        f'{t_stat:.4f}',
        f'{t_stat_w:.4f}',
        f'{u_stat:.0f}',
        f'{chi2:.4f}',
        f'{odds_ratio:.4f}',
        f'{prob_treat_better:.4f}'
    ],
    'P-Value/Prob': [
        f'{p_value_t:.4f}',
        f'{p_value_w:.4f}',
        f'{p_value_u:.4f}',
        f'{p_value_chi2:.4f}',
        f'{p_value_fisher:.4f}',
        f'P(T>C)={prob_treat_better:.4f}'
    ],
    'Effect Size': [
        f'{effect_size_t:.4f}',
        f'{effect_size_t:.4f}',
        f'r={u_stat/(len(y_treat)*len(y_control)):.4f}',
        f'OR={odds_ratio:.4f}',
        f'OR={odds_ratio:.4f}',
        f'DP={effect_size_t:.4f}'
    ],
    'CI Lower': [
        f'{ci_t[0]:.4f}',
        f'{ci_t[0]:.4f}',
        '---',
        '---',
        '---',
        f'{effect_size_t - 1.96*se_t:.4f}'
    ],
    'CI Upper': [
        f'{ci_t[1]:.4f}',
        f'{ci_t[1]:.4f}',
        '---',
        '---',
        '---',
        f'{effect_size_t + 1.96*se_t:.4f}'
    ],
    'Significant': [
        'Yes' if p_value_t < 0.05 else 'No',
        'Yes' if p_value_w < 0.05 else 'No',
        'Yes' if p_value_u < 0.05 else 'No',
        'Yes' if p_value_chi2 < 0.05 else 'No',
        'Yes' if p_value_fisher < 0.05 else 'No',
        'Yes' if abs(prob_treat_better - 0.5) > 0.05 else 'No'
    ]
})

print("\n=== COMPREHENSIVE METHOD COMPARISON (CONVERSION) ===\n")
print(comparison_methods.to_string(index=False))

# Save
comparison_methods.to_csv('../data/outputs/nb10/nb10_method_comparison_conversion.csv', index=False)


=== COMPREHENSIVE METHOD COMPARISON (CONVERSION) ===

                Method Test Stat  P-Value/Prob Effect Size CI Lower CI Upper Significant
    Traditional t-test    7.3897        0.0000      0.0068   0.0050   0.0086         Yes
        Welch's t-test    7.3897        0.0000      0.0068   0.0050   0.0086         Yes
        Mann-Whitney U 228528095        0.0000    r=0.5034      ---      ---         Yes
            Chi-square   53.7902        0.0000   OR=2.2035      ---      ---         Yes
        Fisher's Exact    2.2035        0.0000   OR=2.2035      ---      ---         Yes
Bayesian Beta-Binomial    1.0000 P(T>C)=1.0000   DP=0.0068   0.0050   0.0086         Yes


In [4]:
# Repeat for spend (continuous outcome)
y_spend_treat = men_email['spend']
y_spend_control = control['spend']

# Parametric tests
t_stat_spend, p_value_spend_t = stats.ttest_ind(y_spend_treat, y_spend_control, equal_var=False)
effect_size_spend = y_spend_treat.mean() - y_spend_control.mean()
se_spend = np.sqrt(y_spend_treat.var()/len(y_spend_treat) + y_spend_control.var()/len(y_spend_control))
ci_spend = (effect_size_spend - 1.96*se_spend, effect_size_spend + 1.96*se_spend)

# Non-parametric
u_stat_spend, p_value_spend_u = stats.mannwhitneyu(y_spend_treat, y_spend_control, alternative='two-sided')

# Bayesian (normal likelihood)
mean_t = y_spend_treat.mean()
std_t = y_spend_treat.std()
n_t = len(y_spend_treat)

mean_c = y_spend_control.mean()
std_c = y_spend_control.std()
n_c = len(y_spend_control)

# Posterior for difference
df_bayes = n_t + n_c - 2
t_crit = stats.t.ppf(0.975, df_bayes)

comparison_spend = pd.DataFrame({
    'Method': [
        "Welch's t-test",
        'Mann-Whitney U',
        'Bayesian Normal'
    ],
    'Test Stat': [
        f'{t_stat_spend:.4f}',
        f'{u_stat_spend:.0f}',
        f'{effect_size_spend:.2f}'
    ],
    'P-Value': [
        f'{p_value_spend_t:.4f}',
        f'{p_value_spend_u:.4f}',
        f'{p_value_spend_t:.4f}'
    ],
    'Effect Size': [
        f'${effect_size_spend:.2f}',
        f'r={u_stat_spend/(len(y_spend_treat)*len(y_spend_control)):.4f}',
        f'${effect_size_spend:.2f}'
    ],
    'CI Lower': [
        f'${ci_spend[0]:.2f}',
        '---',
        f'${ci_spend[0]:.2f}'
    ],
    'CI Upper': [
        f'${ci_spend[1]:.2f}',
        '---',
        f'${ci_spend[1]:.2f}'
    ],
    'Significant': [
        'Yes' if p_value_spend_t < 0.05 else 'No',
        'Yes' if p_value_spend_u < 0.05 else 'No',
        'Yes' if p_value_spend_t < 0.05 else 'No'
    ]
})

print("\n=== COMPREHENSIVE METHOD COMPARISON (SPEND) ===\n")
print(comparison_spend.to_string(index=False))

comparison_spend.to_csv('../data/outputs/nb10/nb10_method_comparison_spend.csv', index=False)


=== COMPREHENSIVE METHOD COMPARISON (SPEND) ===

         Method Test Stat P-Value Effect Size CI Lower CI Upper Significant
 Welch's t-test    5.3001  0.0000       $0.77    $0.49    $1.05         Yes
 Mann-Whitney U 228527081  0.0000    r=0.5034      ---      ---         Yes
Bayesian Normal      0.77  0.0000       $0.77    $0.49    $1.05         Yes


## Agreement Analysis

Do different statistical methods agree? Where do they disagree?

**High agreement** -> Results are robust
**Low agreement** -> Results depend on method choice (be cautious)

In [5]:
# Concordance matrix: pairwise agreement between methods
methods_list = comparison_methods['Method'].tolist()
p_values_list = [p_value_t, p_value_w, p_value_u, p_value_chi2, p_value_fisher]
conclusions = [p < 0.05 for p in p_values_list]

# Create concordance matrix
n_methods = len(p_values_list)
concordance_matrix = np.zeros((n_methods, n_methods))

for i in range(n_methods):
    for j in range(n_methods):
        if i == j:
            concordance_matrix[i, j] = 1.0
        else:
            # Both significant or both not significant
            if conclusions[i] == conclusions[j]:
                concordance_matrix[i, j] = 1.0
            else:
                concordance_matrix[i, j] = 0.0

# Plot heatmap

agreement_pct = np.mean(concordance_matrix[np.triu_indices_from(concordance_matrix, k=1)])
print(f"\nOverall agreement across methods: {agreement_pct:.1%}")
print(f"Interpretation: High agreement means robust results.")


Overall agreement across methods: 100.0%
Interpretation: High agreement means robust results.


## Statistical Power Analysis

**Power:** The probability of detecting a true effect if it exists.

Post-hoc power analysis tells us: "Given our sample size and observed effect, what was our ability to detect effects of various sizes?"

**Sample size calculator:** How many customers would we need for 80% power for different effect sizes?

In [6]:
from scipy.stats import norm

# Post-hoc power analysis for conversion
p_treat = y_treat.mean()
p_control = y_control.mean()
n_treat = len(y_treat)
n_control = len(y_control)

# Effect size for proportions (Cohens h)
cohens_h = 2 * (np.arcsin(np.sqrt(p_treat)) - np.arcsin(np.sqrt(p_control)))
print(f"=== POST-HOC POWER ANALYSIS (Conversion) ===")
print(f"\nObserved effect size (Cohens h): {cohens_h:.4f}")
print(f"Treatment conversion rate: {p_treat:.4f}")
print(f"Control conversion rate: {p_control:.4f}")
print(f"Sample size per group: {n_treat}")

# Post-hoc power
z_alpha = norm.ppf(0.975)
ncp = np.sqrt(n_treat + n_control) / 2 * cohens_h
post_hoc_power = 1 - norm.cdf(z_alpha - ncp)

print(f"\nPost-hoc Power (alpha=0.05, two-tailed): {post_hoc_power:.1%}")

# Sample size needed for different effect sizes
print(f"\n=== SAMPLE SIZE CALCULATOR ===")
print(f"\nTo achieve 80% power to detect effect sizes:")

effect_sizes_h = np.array([0.1, 0.2, 0.3, 0.5])
power_target = 0.80
z_beta = norm.ppf(power_target)

sample_size_table = []
for h in effect_sizes_h:
    n_needed = (z_alpha + z_beta)**2 / (2 * h**2)
    sample_size_table.append({
        'Effect Size (h)': f'{h:.2f}',
        'Sample/Group': int(n_needed),
        'Total': int(n_needed * 2)
    })

sample_size_df = pd.DataFrame(sample_size_table)
print(sample_size_df.to_string(index=False))

# Visualize power curve
alphas = np.linspace(0.01, 0.20, 100)
powers_by_alpha = []
for a in alphas:
    ncp_a = np.sqrt(n_treat + n_control) / 2 * a
    power_a = 1 - norm.cdf(z_alpha - ncp_a)
    powers_by_alpha.append(power_a)

=== POST-HOC POWER ANALYSIS (Conversion) ===

Observed effect size (Cohens h): 0.0729
Treatment conversion rate: 0.0125
Control conversion rate: 0.0057
Sample size per group: 21307

Post-hoc Power (alpha=0.05, two-tailed): 100.0%

=== SAMPLE SIZE CALCULATOR ===

To achieve 80% power to detect effect sizes:
Effect Size (h)  Sample/Group  Total
           0.10           392    784
           0.20            98    196
           0.30            43     87
           0.50            15     31


## Assumption Diagnostics

Statistical tests rely on assumptions:

1. **Normality** (for t-tests): Outcomes normally distributed
2. **Homogeneity of Variance**: Variance is same in both groups
3. **Independence**: Observations are independent
4. **Sample Ratio Mismatch (SRM)**: Did randomization work?

In [7]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Sensitivity Analysis

How robust are our conclusions to different assumptions and choices?

1. **Significance level**: What if we used alpha=0.10 instead of 0.05?
2. **One-sided vs two-sided**: What if we only cared about increase?
3. **Outlier removal**: What if we excluded extreme values?

In [8]:
# Sensitivity analysis for conversion
alpha_values = [0.01, 0.05, 0.10]
sensitivity_results = []

for alpha in alpha_values:
    z_crit = norm.ppf(1 - alpha/2)
    ci_width = z_crit * se_t
    p_sig = p_value_t < alpha
    
    sensitivity_results.append({
        'Significance Level': f'{alpha:.2f}',
        'Critical t': f'{z_crit:.3f}',
        'Significant': 'Yes' if p_sig else 'No',
        'Effect': f'{effect_size_t:.4f}'
    })

# One-sided
p_one_sided = p_value_t / 2
sensitivity_results.append({
    'Significance Level': '0.05 (1-sided)',
    'Critical t': '1.645',
    'Significant': 'Yes' if p_one_sided < 0.05 else 'No',
    'Effect': f'{effect_size_t:.4f}'
})

# Outlier removal
y_treat_trim = y_treat[(y_treat >= y_treat.quantile(0.01)) & (y_treat <= y_treat.quantile(0.99))]
y_control_trim = y_control[(y_control >= y_control.quantile(0.01)) & (y_control <= y_control.quantile(0.99))]
t_trim, p_trim = stats.ttest_ind(y_treat_trim, y_control_trim, equal_var=False)
effect_trim = y_treat_trim.mean() - y_control_trim.mean()

sensitivity_results.append({
    'Significance Level': '0.05 (trimmed 1%)',
    'Critical t': f'{t_trim:.3f}',
    'Significant': 'Yes' if p_trim < 0.05 else 'No',
    'Effect': f'{effect_trim:.4f}'
})

sensitivity_df = pd.DataFrame(sensitivity_results)
print("\n=== SENSITIVITY ANALYSIS ===\n")
print(sensitivity_df.to_string(index=False))


=== SENSITIVITY ANALYSIS ===

Significance Level Critical t Significant Effect
              0.01      2.576         Yes 0.0068
              0.05      1.960         Yes 0.0068
              0.10      1.645         Yes 0.0068
    0.05 (1-sided)      1.645         Yes 0.0068
 0.05 (trimmed 1%)     16.443         Yes 0.0125


## Practical vs Statistical Significance

**Statistical significance** is not the same as **business significance**.

A difference can be:
- Statistically significant AND practically meaningful
- Statistically significant BUT negligible for business
- NOT statistically significant BUT practically important

In [9]:
# Business impact analysis for conversion
print("=== BUSINESS IMPACT ANALYSIS ===\n")

email_cost = 0.50
conversion_value = 50.0
n_customers = 1_000_000
percent_receiving_email = 0.30

customers_receiving = int(n_customers * percent_receiving_email)
customers_control = n_customers - customers_receiving

conversions_email = customers_receiving * p_treat
conversions_control = customers_receiving * p_control

incremental_conversions = conversions_email - conversions_control
total_spend = customers_receiving * email_cost
total_incremental_revenue = incremental_conversions * conversion_value
net_profit = total_incremental_revenue - total_spend

print(f"Scenario: Send emails to {percent_receiving_email:.0%} of {n_customers/1e6:.1f}M customers")
print(f"\nObserved Effect:")
print(f"  Treatment conversion rate: {p_treat:.3f} ({p_treat*100:.1f}%)")
print(f"  Control conversion rate: {p_control:.3f} ({p_control*100:.1f}%)")
print(f"  Incremental lift: {effect_size_t*100:.2f} percentage points")

print(f"\nFinancial Impact:")
print(f"  Customers emailed: {customers_receiving:,.0f}")
print(f"  Total email cost: ${total_spend:,.0f}")
print(f"  Incremental conversions: {incremental_conversions:,.0f}")
print(f"  Incremental revenue: ${total_incremental_revenue:,.0f}")
print(f"  Net profit: ${net_profit:,.0f}")

print(f"\nROI Analysis:")
roi = (net_profit / total_spend) * 100 if total_spend > 0 else 0
print(f"  ROI: {roi:.0f}%")

print(f"\nDecision:")
if net_profit > 0 and p_value_t < 0.05:
    print(f"  PROFITABLE AND STATISTICALLY SIGNIFICANT - Implement")
elif net_profit > 0:
    print(f"  PROFITABLE BUT NOT SIGNIFICANT - Run larger test")
else:
    print(f"  NOT PROFITABLE - Do not implement")

=== BUSINESS IMPACT ANALYSIS ===

Scenario: Send emails to 30% of 1.0M customers

Observed Effect:
  Treatment conversion rate: 0.013 (1.3%)
  Control conversion rate: 0.006 (0.6%)
  Incremental lift: 0.68 percentage points

Financial Impact:
  Customers emailed: 300,000
  Total email cost: $150,000
  Incremental conversions: 2,042
  Incremental revenue: $102,075
  Net profit: $-47,925

ROI Analysis:
  ROI: -32%

Decision:
  NOT PROFITABLE - Do not implement


In [10]:
# Save master results
results_summary = pd.DataFrame({
    'Test': ['Conversion (t-test)', 'Spend (Welch)', 'Conversion (Chi-sq)'],
    'Effect Size': [f'{effect_size_t:.4f}', f'{effect_size_spend:.2f}', f'{odds_ratio:.4f}'],
    'P-Value': [f'{p_value_t:.4f}', f'{p_value_spend_t:.4f}', f'{p_value_chi2:.4f}'],
    'Significant': [
        'Yes' if p_value_t < 0.05 else 'No',
        'Yes' if p_value_spend_t < 0.05 else 'No',
        'Yes' if p_value_chi2 < 0.05 else 'No'
    ]
})

results_summary.to_csv('../data/outputs/nb10/nb10_comprehensive_results_summary.csv', index=False)

print("\nFiles saved:")
print("1. method_comparison_conversion.csv")
print("2. method_comparison_spend.csv")
print("3. comprehensive_results_summary.csv")
print("\nVisualizations:")
print("4. method_agreement.png")
print("5. power_analysis.png")
print("6. normality_qq_plots.png")
print("7. sensitivity_analysis.png")


Files saved:
1. method_comparison_conversion.csv
2. method_comparison_spend.csv
3. comprehensive_results_summary.csv

Visualizations:
4. method_agreement.png
5. power_analysis.png
6. normality_qq_plots.png
7. sensitivity_analysis.png


## Key Takeaways

### Notebook Summary

This notebook synthesized all A/B testing methods:
1. **Traditional t-tests** - statistically rigorous
2. **CUPED** - reduces variance with covariates
3. **Uplift modeling** - identifies individual treatment effects
4. **Multi-armed bandits** - adaptive allocation
5. **Bayesian approach** - sequential analysis

### Method Agreement
High concordance across multiple statistical tests indicates robust results.

### Power Analysis
Post-hoc power reveals our ability to detect effects of various sizes.

### Assumptions
Diagnostic tests ensure our statistical conclusions are valid.

### Business Impact
ROI analysis connects statistical significance to business value.

### Recommendations

Choose your method based on:
- **Outcome type** (binary vs continuous)
- **Decision timeline** (one-time vs ongoing)
- **Cost constraints** (sample size vs quality)
- **Personalization needs** (average vs individual effects)

---

## Blog-Ready Plotly Charts

The cells below regenerate the charts from this notebook as responsive Plotly
HTML files for embedding in the blog post. They are **self-contained**: each
one re-loads the clean dataset from nb01 and re-derives the statistics it
needs, so you can run this section in isolation.

Outputs are written to `data/outputs/nb##/` with the suffix `_interactive.html`.

**Required packages:** `plotly` (install with `pip install plotly` if missing).

In [11]:
# ============================================================
# Blog-Ready Plotly Charts — self-contained, embed-friendly
# ============================================================
# These cells produce responsive Plotly HTML files for the blog post.
# They re-load from the nb01 clean CSV and re-derive stats so the section
# runs standalone. Each figure uses:
#   - include_plotlyjs='cdn' (single shared CDN load on the blog page)
#   - config={'responsive': True} so it resizes to container width
#   - automargin=True on axes + generous margins so labels never clip
#   - rotated tick labels on long categories, headroom for outside labels
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # inline-render Plotly in cell output

OUT_DIR = os.path.abspath("../data/outputs/nb10")
os.makedirs(OUT_DIR, exist_ok=True)
CLEAN_CSV = os.path.abspath("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df_blog = pd.read_csv(CLEAN_CSV)

# Shared palette aligned with nb01 Plotly charts
COLORS = {
    "Mens E-Mail": "#4C8BB8", "Womens E-Mail": "#5FA85F", "No E-Mail": "#E89B4C",
    "Match": "#2ECC71", "Mismatch": "#E74C3C", "Mixed": "#F39C12", "Control": "#95A5A6",
    "Treatment (Any Email)": "#4C8BB8",
}
PLOTLY_KW = dict(include_plotlyjs="cdn", full_html=True,
                 config={"responsive": True, "displaylogo": False})
BASE_LAYOUT = dict(template="plotly_white",
                   font=dict(family="Arial, sans-serif", size=13),
                   title_x=0.5,
                   margin=dict(l=70, r=40, t=90, b=90),
                   hoverlabel=dict(bgcolor="white", font_size=12))
print(f"Blog-ready Plotly charts will be written to: {OUT_DIR}")


Blog-ready Plotly charts will be written to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb10


In [13]:
from scipy import stats as spstats
# Chart 1: Method-agreement heatmap across methods for the primary question
# Methods compute a single p-value or probability of superiority for Mens vs Control on conversion
from scipy.stats import chi2_contingency as _c2, ttest_ind as _ttest
m_conv = df_blog[df_blog["segment"]=="Mens E-Mail"]["conversion"]
c_conv = df_blog[df_blog["segment"]=="No E-Mail"]["conversion"]
# 1) Z-test (two proportions)
p1, n1 = m_conv.mean(), len(m_conv); p2, n2 = c_conv.mean(), len(c_conv)
pool = (m_conv.sum()+c_conv.sum())/(n1+n2)
se = np.sqrt(pool*(1-pool)*(1/n1 + 1/n2)); z = (p1-p2)/se
p_ztest = 2*(1 - spstats.norm.cdf(abs(z)))
# 2) Chi-square
ct2 = pd.crosstab(df_blog[df_blog["segment"].isin(["Mens E-Mail","No E-Mail"])]["segment"],
                  df_blog[df_blog["segment"].isin(["Mens E-Mail","No E-Mail"])]["conversion"])
chi2_, p_chi, *_ = _c2(ct2.values)
# 3) Bootstrap p-value via 2000 reps
np.random.seed(42); t = m_conv.values; c = c_conv.values
obs = t.mean() - c.mean()
combined = np.concatenate([t, c])
null_diffs = np.empty(2000)
for i in range(2000):
    np.random.shuffle(combined)
    null_diffs[i] = combined[:n1].mean() - combined[n1:].mean()
p_boot = (np.abs(null_diffs) >= abs(obs)).mean()
# 4) Bayesian prob not-best (≈ 1 - P(M>C))
psm = np.random.beta(1+m_conv.sum(), 1+n1-m_conv.sum(), size=20000)
psc = np.random.beta(1+c_conv.sum(), 1+n2-c_conv.sum(), size=20000)
prob_m_beats_c = float((psm > psc).mean())

summary = pd.DataFrame({
    "Method": ["Z-test", "Chi-square", "Bootstrap permutation", "Bayesian P(M>C)"],
    "p-value / prob": [p_ztest, p_chi, p_boot, prob_m_beats_c],
    "Significant @0.05": [p_ztest<0.05, p_chi<0.05, p_boot<0.05, prob_m_beats_c>0.95],
})
print(summary)

# Agreement matrix: 1 if both methods say significant
sigs = summary["Significant @0.05"].astype(int).values
n_m = len(sigs)
agree = np.ones((n_m, n_m), dtype=int)  # will be 1 if same decision
for i in range(n_m):
    for j in range(n_m):
        agree[i,j] = int(summary["Significant @0.05"].iloc[i] == summary["Significant @0.05"].iloc[j])
fig = go.Figure(go.Heatmap(z=agree, x=summary["Method"], y=summary["Method"],
                           text=[["agree" if v else "disagree" for v in row] for row in agree],
                           texttemplate="%{text}", colorscale=[[0,"#E74C3C"],[1,"#2ECC71"]],
                           showscale=False,
                           hovertemplate="%{y} vs %{x}: %{text}<extra></extra>"))
fig.update_xaxes(automargin=True, tickangle=-20)
fig.update_yaxes(automargin=True)
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=140, r=40, t=90, b=120)},
                  title="Method Agreement on Mens E-Mail vs Control (conversion)", height=460)
fig.write_html(os.path.join(OUT_DIR, "nb10_method_agreement_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb10_method_agreement_interactive.html")

# Chart 2: Cross-method forest plot of effect estimates
# Use all four methods' point estimates and CIs (where applicable)
methods = []
# Z-test estimate + CI
se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2); diff = p1-p2
methods.append(("Z-test", diff, diff-1.96*se_diff, diff+1.96*se_diff))
# Bootstrap CI
np.random.seed(42); boot = np.empty(3000)
for i in range(3000):
    boot[i] = np.random.choice(t, n1, replace=True).mean() - np.random.choice(c, n2, replace=True).mean()
blo, bhi = np.percentile(boot, [2.5, 97.5])
methods.append(("Bootstrap", diff, blo, bhi))
# Bayesian credible interval on difference
crlo, crhi = np.percentile(psm - psc, [2.5, 97.5])
methods.append(("Bayesian", float((psm-psc).mean()), crlo, crhi))

fig = go.Figure()
for i, (name, est, lo, hi) in enumerate(methods):
    fig.add_trace(go.Scatter(x=[lo*100, hi*100], y=[i,i], mode="lines",
                             line=dict(color="#4C8BB8", width=3), showlegend=False,
                             hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=[est*100], y=[i], mode="markers+text",
                             marker=dict(size=14, color="#4C8BB8", line=dict(color="black", width=1)),
                             text=[f"{est*100:+.3f} pp"], textposition="middle right",
                             showlegend=False,
                             hovertemplate=f"<b>{name}</b><br>Est: %{{x:.3f}} pp<br>CI: [{lo*100:.3f}, {hi*100:.3f}]<extra></extra>"))
fig.add_vline(x=0, line_dash="dash", line_color="#555")
fig.update_yaxes(tickvals=list(range(len(methods))),
                 ticktext=[m[0] for m in methods], automargin=True)
fig.update_xaxes(title="Mens − Control conversion (percentage points)", automargin=True)
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=140, r=120, t=90, b=80)},
                  title="Cross-Method Effect-Size Agreement", height=440)
fig.write_html(os.path.join(OUT_DIR, "nb10_cross_method_forest_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb10_cross_method_forest_interactive.html")


                  Method  p-value / prob  Significant @0.05
0                 Z-test    1.523226e-13               True
1             Chi-square    2.230841e-13               True
2  Bootstrap permutation    0.000000e+00               True
3        Bayesian P(M>C)    1.000000e+00               True


  ✓ nb10_method_agreement_interactive.html


  ✓ nb10_cross_method_forest_interactive.html


### Results-Display Tables (embed-ready go.Table cards for every printed output)

Each card below mirrors one of the printed console blocks and saves as its own
HTML file under `../data/outputs/nb10/` so it can be dropped straight into the
blog post.


In [14]:
# Reusable go.Table card helper (reuses OUT_DIR/PLOTLY_KW/BASE_LAYOUT from Plotly setup cell above)
def table_card(title, header_vals, cell_cols, colwidths,
               row_colors=None, cell_font_size=12, height_extra=80, align="center"):
    n_rows = len(cell_cols[0]) if cell_cols else 0
    stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(n_rows)]
    fill = row_colors if row_colors else [stripe for _ in cell_cols]
    fig = go.Figure(data=[go.Table(
        columnwidth=colwidths,
        header=dict(values=[f"<b>{h}</b>" for h in header_vals],
                    fill_color="#2C3E50",
                    font=dict(color="white", size=13),
                    align="center", height=36),
        cells=dict(values=cell_cols, fill_color=fill, align=align,
                   font=dict(size=cell_font_size, family="monospace"),
                   height=30))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                      title=title,
                      height=36 + 30*n_rows + height_extra)
    return fig

def sig_colors(flags):
    return ["#D5F5E3" if f else "#FADBD8" for f in flags]

def stripe_col(n):
    return ["#F8F9F9" if i%2==0 else "white" for i in range(n)]

# Method Comparison & Diagnostics — Results-Display Tables
from scipy import stats as _sp
from scipy.stats import chi2_contingency, fisher_exact

any_email = df_blog[df_blog["segment"] != "No E-Mail"]
control   = df_blog[df_blog["segment"] == "No E-Mail"]
y_t = any_email["conversion"].values; y_c = control["conversion"].values
s_t = any_email["spend"].values;      s_c = control["spend"].values

# --- Method comparison: conversion ---
# t-test (treat binary as continuous), Welch, Mann-Whitney, chi-square, Fisher
t_stat, p_t  = _sp.ttest_ind(y_t, y_c, equal_var=True)
w_stat, p_w  = _sp.ttest_ind(y_t, y_c, equal_var=False)
u_stat, p_u  = _sp.mannwhitneyu(y_t, y_c, alternative="two-sided")
ct = np.array([[y_t.sum(), len(y_t)-y_t.sum()],
               [y_c.sum(), len(y_c)-y_c.sum()]])
chi2_stat, p_chi2, _, _ = chi2_contingency(ct)
odds, p_fisher = fisher_exact(ct)

rows_conv = [
    ("Student's t-test (equal var)",    f"{t_stat:.4f}",  f"{p_t:.3e}",    "Yes" if p_t<0.05 else "No"),
    ("Welch's t-test",                  f"{w_stat:.4f}",  f"{p_w:.3e}",    "Yes" if p_w<0.05 else "No"),
    ("Mann–Whitney U",                  f"{u_stat:.1f}",  f"{p_u:.3e}",    "Yes" if p_u<0.05 else "No"),
    ("Chi-square",                      f"{chi2_stat:.4f}", f"{p_chi2:.3e}", "Yes" if p_chi2<0.05 else "No"),
    ("Fisher's exact",                  f"OR={odds:.3f}", f"{p_fisher:.3e}", "Yes" if p_fisher<0.05 else "No"),
]
sig_conv = [r[3]=="Yes" for r in rows_conv]
fig = go.Figure(data=[go.Table(
    columnwidth=[260, 180, 180, 150],
    header=dict(values=[f"<b>{h}</b>" for h in ["Method","Statistic","P-value","Significant?"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=list(zip(*rows_conv)),
               fill_color=[stripe_col(len(rows_conv))]*3 + [sig_colors(sig_conv)],
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Method Comparison — Conversion (Any Email vs Control)",
                  height=36 + 30*len(rows_conv) + 90)
fig.write_html(os.path.join(OUT_DIR, "nb10_method_comparison_conversion_interactive.html"), **PLOTLY_KW)
fig.show()

# --- Method comparison: spend ---
t_stat, p_t  = _sp.ttest_ind(s_t, s_c, equal_var=True)
w_stat, p_w  = _sp.ttest_ind(s_t, s_c, equal_var=False)
u_stat, p_u  = _sp.mannwhitneyu(s_t, s_c, alternative="two-sided")
rows_spend = [
    ("Student's t-test (equal var)", f"{t_stat:.4f}", f"{p_t:.3e}", "Yes" if p_t<0.05 else "No"),
    ("Welch's t-test",               f"{w_stat:.4f}", f"{p_w:.3e}", "Yes" if p_w<0.05 else "No"),
    ("Mann–Whitney U",               f"{u_stat:.1f}", f"{p_u:.3e}", "Yes" if p_u<0.05 else "No"),
]
sig_s = [r[3]=="Yes" for r in rows_spend]
fig = go.Figure(data=[go.Table(
    columnwidth=[260, 180, 180, 150],
    header=dict(values=[f"<b>{h}</b>" for h in ["Method","Statistic","P-value","Significant?"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=list(zip(*rows_spend)),
               fill_color=[stripe_col(len(rows_spend))]*3 + [sig_colors(sig_s)],
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Method Comparison — Spend (Any Email vs Control)",
                  height=36 + 30*len(rows_spend) + 90)
fig.write_html(os.path.join(OUT_DIR, "nb10_method_comparison_spend_interactive.html"), **PLOTLY_KW)
fig.show()

# --- Post-hoc power + sample-size calculator ---
p_t_rate = y_t.mean(); p_c_rate = y_c.mean()
h = 2*(np.arcsin(np.sqrt(p_t_rate)) - np.arcsin(np.sqrt(p_c_rate)))
z_alpha = _sp.norm.ppf(0.975)
ncp = np.sqrt(len(y_t)+len(y_c))/2 * h
post_power = 1 - _sp.norm.cdf(z_alpha - ncp)

z_beta = _sp.norm.ppf(0.80)
ss_rows = []
for eh in [0.1, 0.2, 0.3, 0.5]:
    n = (z_alpha + z_beta)**2 / (2*eh**2)
    ss_rows.append((f"{eh:.2f}", f"{int(n):,}", f"{int(2*n):,}"))

fig = table_card(
    f"Post-Hoc Power & Sample-Size Calculator (Observed Cohen's h = {h:.4f}, post-hoc power = {post_power:.1%})",
    ["Target effect size h", "Sample per arm (80% power, α=0.05)", "Total N"],
    [[r[0] for r in ss_rows], [r[1] for r in ss_rows], [r[2] for r in ss_rows]],
    [200, 320, 200])
fig.write_html(os.path.join(OUT_DIR, "nb10_power_sample_size_interactive.html"), **PLOTLY_KW)
fig.show()

# --- Sensitivity analysis ---
rows_sens = []
se_t = np.sqrt(y_t.var(ddof=1)/len(y_t) + y_c.var(ddof=1)/len(y_c))
effect = y_t.mean() - y_c.mean()
tstat_c, pval_c = _sp.ttest_ind(y_t, y_c, equal_var=False)
for alpha in [0.01, 0.05, 0.10]:
    z_c = _sp.norm.ppf(1 - alpha/2)
    rows_sens.append((f"α = {alpha:.2f} (two-sided)", f"{z_c:.3f}",
                      "Yes" if pval_c<alpha else "No", f"{effect:.5f}"))
rows_sens.append(("α = 0.05 (one-sided)", "1.645",
                  "Yes" if pval_c/2 < 0.05 else "No", f"{effect:.5f}"))
# trimmed
lo_t, hi_t = np.quantile(y_t, [0.01, 0.99])
lo_c, hi_c = np.quantile(y_c, [0.01, 0.99])
y_t_tr = y_t[(y_t>=lo_t)&(y_t<=hi_t)]
y_c_tr = y_c[(y_c>=lo_c)&(y_c<=hi_c)]
tt_tr, pp_tr = _sp.ttest_ind(y_t_tr, y_c_tr, equal_var=False)
rows_sens.append(("α = 0.05 (trimmed 1%)", f"{tt_tr:.3f}",
                  "Yes" if pp_tr<0.05 else "No",
                  f"{y_t_tr.mean()-y_c_tr.mean():.5f}"))
sig_sens = [r[2]=="Yes" for r in rows_sens]
fig = go.Figure(data=[go.Table(
    columnwidth=[260, 160, 150, 160],
    header=dict(values=[f"<b>{h}</b>" for h in ["Scenario","Critical value / t","Significant?","Effect"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=list(zip(*rows_sens)),
               fill_color=[stripe_col(len(rows_sens))]*2 + [sig_colors(sig_sens)] + [stripe_col(len(rows_sens))],
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Sensitivity Analysis — Conversion (Any Email vs Control)",
                  height=36 + 30*len(rows_sens) + 90)
fig.write_html(os.path.join(OUT_DIR, "nb10_sensitivity_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb10 method-comparison / power / sensitivity cards saved")


  ✓ nb10 method-comparison / power / sensitivity cards saved
